# Feature Engineering

**OJO: Al finalizar este notebook van a ver, varias clases a predecir llamadas "clase_...". Cuando se seleccione para entrenar se debe seleccionar una y descartar las otras. No incluirlas en las features de entrenamiento!!**

# Modulo 2

In [ ]:
import polars as pl

# Cargue las decisiones del dataset en el módulo 1
Agrupamiento = "grpClienteProducto"  # "grpProducto"|"grpClienteProducto"
Completado = "fill0"  # "fill0"|"fillNA"
Formato = "denseLife"  # "denseLife"|"denseFull"
Productos = ""  # "_tgtFilter"|"" (Todos los productos)

# Se elige el nombre del dataset con el que se quiere trabajar
dataset = f"preprocesado_{Agrupamiento}_{Completado}_{Formato}{Productos}.parquet"

# Ruta del bucket que contiene los datasets (Se podría modificar para trabajar por experimento)
Ruta = f"/content/buckets/b1/datasets/"

# Leer archivo que sale del módulo 1
df_final = pl.read_parquet(Ruta+dataset)

# Convertimos la columna periodo a formato entero YYYYMM
df_final = df_final.with_columns(
    (pl.col("periodo").dt.year() * 100 + pl.col("periodo").dt.month()).alias("periodo")
)

# Ver el dataset
print(df_final)

shape: (10_792_694, 14)
┌──────────────┬─────────────┬────────────┬─────────┬───┬─────────┬───────┬──────────┬─────────────┐
│ Agrupacion_I ┆ customer_id ┆ product_id ┆ periodo ┆ … ┆ cat3    ┆ brand ┆ sku_size ┆ descripcion │
│ D            ┆ ---         ┆ ---        ┆ ---     ┆   ┆ ---     ┆ ---   ┆ ---      ┆ ---         │
│ ---          ┆ i64         ┆ i64        ┆ i32     ┆   ┆ str     ┆ str   ┆ i64      ┆ str         │
│ i64          ┆             ┆            ┆         ┆   ┆         ┆       ┆          ┆             │
╞══════════════╪═════════════╪════════════╪═════════╪═══╪═════════╪═══════╪══════════╪═════════════╡
│ 1004620876   ┆ 10046       ┆ 20876      ┆ 201701  ┆ … ┆ Acond   ┆ NIVEA ┆ 200      ┆ Cabello     │
│              ┆             ┆            ┆         ┆   ┆ Mujer   ┆       ┆          ┆ anti caída  │
│ 1004620876   ┆ 10046       ┆ 20876      ┆ 201702  ┆ … ┆ Acond   ┆ NIVEA ┆ 200      ┆ Cabello     │
│              ┆             ┆            ┆         ┆   ┆ Mujer   ┆

In [2]:
df_final.columns

['Agrupacion_ID',
 'customer_id',
 'product_id',
 'periodo',
 'tn',
 'cust_request_qty',
 'cust_request_tn',
 'plan_precios_cuidados',
 'cat1',
 'cat2',
 'cat3',
 'brand',
 'sku_size',
 'descripcion']

## Funciones

### Lagear dataset (panel supervisado)

In [ ]:
def lagear_dataset(df, granularidad=None, periodo_final=201912, n_periodos=42,
                   max_lags=24, horizonte=2, keys=None):
    """
    Arma la tabla supervisada (panel): una fila por (clave, periodo actual).

    Para CADA periodo t del rango [periodo_final - (n_periodos - 1) .. periodo_final]
    (los ultimos `n_periodos` meses hasta `periodo_final`) genera, por clave:
      - tn0..tn{max_lags}: tn del propio t y de los meses anteriores (t, t-1, ..., t-max_lags).
      - clase_tn        : tn `horizonte` meses adelante -> tn(t + horizonte). Default 2.
      - ppc0            : plan_precios_cuidados en el propio t (si viene la columna).
      - periodo         : el periodo actual t (clave temporal de cada fila).

    A diferencia del achatado viejo (un unico t0 fijo), aca t recorre un rango: cada
    mes es su propio "periodo actual". Solo se genera fila para los (clave, t) con
    observacion en t; los lags que caen fuera de la serie quedan NULL. Los 0 dentro de
    la vida vienen del completado del modulo 1.

    Los lags se calculan con shift sobre la serie ordenada por mes: asume que la serie
    es densa mensual dentro de la vida de cada clave (garantizado por el completado del
    modulo 1: denseLife / denseFull), de modo que un shift de k filas == k meses.

    Params
    ------
    df           : long con periodo (AAAAMM entero), tn y las columnas de `keys`.
    granularidad : "p" (producto) o "pc" (producto-cliente). Atajo de `keys`.
    periodo_final: ultimo periodo actual (AAAAMM). Default 201910.
    n_periodos   : cuantos periodos actuales generar hacia atras. Default 42.
    max_lags     : cantidad de lags -> tn0..tn{max_lags}.
    horizonte    : cuantos meses adelante es la clase. Default 2.
    keys         : columnas de agrupacion; si viene, gana sobre `granularidad`.
    """
    if keys is None:
        if granularidad == "pc":
            keys = ["product_id", "customer_id"]
        elif granularidad == "p":
            keys = ["product_id"]
        else:
            raise ValueError(
                f"granularidad no soportada: {granularidad!r}. Usa 'p' / 'pc', o pasa keys=[...]"
            )

    m_final = (periodo_final // 100) * 12 + (periodo_final % 100)
    m_ini = m_final - (n_periodos - 1)
    has_ppc = "plan_precios_cuidados" in df.columns

    # Serie por (clave, periodo), ordenada cronologicamente dentro de cada clave.
    # En "p" agrega tn sobre los clientes; en "pc" es 1:1.
    aggs = [pl.col("tn").sum().alias("tn")]
    if has_ppc:
        aggs.append(pl.col("plan_precios_cuidados").max().alias("ppc"))
    serie = (
        df.group_by(keys + ["periodo"]).agg(aggs)
          .with_columns(((pl.col("periodo") // 100) * 12 + (pl.col("periodo") % 100)).alias("m"))
          .sort(keys + ["m"])
    )

    # Lags por shift (serie densa -> shift de k filas == k meses). shift(k) mira k meses
    # atras; shift(-horizonte) mira horizonte meses adelante (la clase). En los bordes
    # (previo al first_sell / posterior al ultimo mes) shift devuelve NULL. over(keys)
    # evita que el shift cruce de una clave a otra.
    exprs = [pl.col("tn").shift(k).over(keys).alias(f"tn{k}") for k in range(max_lags + 1)]
    exprs.append(pl.col("tn").shift(-horizonte).over(keys).alias("clase_tn"))
    serie = serie.with_columns(exprs)

    # Panel: periodos "actuales" (rango) con observacion propia (tn0 no nulo, descarta
    # meses fuera de la vida en datasets densos tipo denseFull; en denseLife es no-op).
    panel = serie.filter(
        (pl.col("m") >= m_ini) & (pl.col("m") <= m_final) & pl.col("tn0").is_not_null()
    )

    extra = []
    if has_ppc:
        panel = panel.with_columns(pl.col("ppc").alias("ppc0"))
        extra = ["ppc0"]

    # Metadata 1:1 con la clave (atributos del producto): la arrastramos.
    meta_cols = [c for c in ["Agrupacion_ID", "cat1", "cat2", "cat3", "brand"]
                 if c in df.columns and c not in keys]
    id_cols = []
    if meta_cols:
        meta = df.select(keys + meta_cols).unique(subset=keys)
        panel = panel.join(meta, on=keys, how="left")
        id_cols = meta_cols

    lag_cols = [f"tn{k}" for k in range(max_lags + 1)]
    return (
        panel
        .select(id_cols + keys + ["periodo"] + lag_cols + extra + ["clase_tn"])
        .sort(keys + ["periodo"])
    )

### Normalizar achatado

In [4]:
def normalizar_achatado(df_achatado, metodo="recta", norm_lags=None,
                        prefix="", clase_col="clase_tn", b0_name="B0", b1_name="B1"):
    """
    Normaliza fila a fila las columnas {prefix}tn* (y opcionalmente {clase_col})
    de df_achatado.

    Los parametros de normalizacion (B0, B1) se calculan SOLO con la ventana
    {prefix}tn0..{prefix}tn{norm_lags} (nunca con la clase -> evita leakage) y
    luego se aplican a todos los {prefix}tn* y a la clase, generando
    {prefix}tn0_norm..{prefix}tnN_norm y {clase_col}_norm.

    Params
    ------
    metodo:
      "recta"   -> ajuste lineal por minimos cuadrados sobre la ventana.
                   B0 = ordenada al origen, B1 = pendiente.
                   norm = valor - (B0 + B1 * lag)   (la clase esta en lag = -2)
      "zscore"  -> B0 = media, B1 = desvio.   norm = (valor - B0) / B1
      "minmax"  -> B0 = min,   B1 = max-min.  norm = (valor - B0) / B1
      "media"   -> B0 = 0,     B1 = media.    norm = valor / B1
    norm_lags:
      hasta que lag usar para calcular B0/B1 (ventana {prefix}tn0..{prefix}tn{norm_lags}).
      None = usa todos los lags disponibles.
    prefix:
      prefijo de las columnas de lags a normalizar (ej. "" para pc, "p_" para producto).
    clase_col:
      nombre de la columna de clase a normalizar; None si no hay clase (ej. features
      de producto, donde no se normaliza ninguna clase).
    b0_name, b1_name:
      nombres con los que se guardan los parametros de normalizacion (ej. "B0"/"B1"
      para pc, "p_B0"/"p_B1" para producto).

    Devuelve df_achatado + columnas b0_name, b1_name, {prefix}tn*_norm y (si aplica)
    {clase_col}_norm.
    """
    p = len(prefix)
    lag_cols = sorted(
        [c for c in df_achatado.columns if c.startswith(prefix + "tn") and c[p + 2:].isdigit()],
        key=lambda c: int(c[p + 2:]),
    )
    n_max = int(lag_cols[-1][p + 2:])
    if norm_lags is None:
        norm_lags = n_max
    fit_cols = [f"{prefix}tn{k}" for k in range(norm_lags + 1)]

    if metodo == "recta":
        # Minimos cuadrados sobre la ventana, ignorando los nulls
        n   = pl.sum_horizontal([pl.col(c).is_not_null().cast(pl.Float64) for c in fit_cols])
        Sy  = pl.sum_horizontal([pl.col(c) for c in fit_cols])
        Sx  = pl.sum_horizontal([pl.when(pl.col(c).is_not_null()).then(float(k)).otherwise(0.0)
                                 for k, c in enumerate(fit_cols)])
        Sxx = pl.sum_horizontal([pl.when(pl.col(c).is_not_null()).then(float(k * k)).otherwise(0.0)
                                 for k, c in enumerate(fit_cols)])
        Sxy = pl.sum_horizontal([pl.when(pl.col(c).is_not_null()).then(pl.col(c) * float(k)).otherwise(0.0)
                                 for k, c in enumerate(fit_cols)])
        denom = n * Sxx - Sx * Sx
        slope = pl.when(denom != 0).then((n * Sxy - Sx * Sy) / denom).otherwise(0.0)
        inter = pl.when(n > 0).then((Sy - slope * Sx) / n).otherwise(None)

        df = df_achatado.with_columns(inter.alias(b0_name), slope.alias(b1_name))

        # norm = valor - (B0 + B1 * lag);  la clase esta en lag = -2 (t0 + 2)
        exprs = [
            (pl.col(c) - (pl.col(b0_name) + pl.col(b1_name) * float(int(c[p + 2:])))).alias(f"{c}_norm")
            for c in lag_cols
        ]
        if clase_col is not None:
            exprs.append((pl.col(clase_col) - (pl.col(b0_name) + pl.col(b1_name) * (-2.0))).alias(f"{clase_col}_norm"))
        return df.with_columns(exprs)

    # --- metodos afines: norm = (valor - B0) / B1 ---
    if metodo == "zscore":
        B0 = pl.mean_horizontal(fit_cols)
        B1 = pl.concat_list(fit_cols).list.std()
    elif metodo == "minmax":
        B0 = pl.min_horizontal(fit_cols)
        B1 = pl.max_horizontal(fit_cols) - pl.min_horizontal(fit_cols)
    elif metodo == "media":
        B0 = pl.lit(0.0)
        B1 = pl.mean_horizontal(fit_cols)
    else:
        raise ValueError(f"metodo no soportado: {metodo}")

    df = df_achatado.with_columns(B0.alias(b0_name), B1.alias(b1_name))
    # B1 seguro para no dividir por 0 ni por null (series constantes / sin datos)
    b1_safe = pl.when((pl.col(b1_name) == 0) | pl.col(b1_name).is_null()).then(1.0).otherwise(pl.col(b1_name))
    exprs = [((pl.col(c) - pl.col(b0_name)) / b1_safe).alias(f"{c}_norm") for c in lag_cols]
    if clase_col is not None:
        exprs.append(((pl.col(clase_col) - pl.col(b0_name)) / b1_safe).alias(f"{clase_col}_norm"))
    return df.with_columns(exprs)



### Desnormalizar

In [5]:
def desnormalizar(df, col_norm, metodo, lag=-2, alias="pred_tn"):
    """
    Recupera la escala original (toneladas) de una columna normalizada,
    usando los parametros B0/B1 guardados por normalizar_achatado.
    Es el inverso exacto de normalizar_achatado.

    Params
    ------
    df       : DataFrame que tiene las columnas B0, B1 y col_norm.
    col_norm : columna normalizada a desnormalizar (ej. la prediccion del modelo).
    metodo   : el MISMO metodo usado al normalizar ("recta"|"zscore"|"minmax"|"media").
    lag      : posicion temporal de la columna respecto de t0.
               clase_tn (y su prediccion) esta en lag = -2 (t0 + 2); tn{k} esta en lag = k.
    alias    : nombre de la columna de salida.

    Devuelve df + columna 'alias' en escala original.
    """
    if metodo == "recta":
        # inverso de: norm = valor - (B0 + B1 * lag)
        valor = pl.col(col_norm) + (pl.col("B0") + pl.col("B1") * float(lag))
    elif metodo in ("zscore", "minmax", "media"):
        # inverso de: norm = (valor - B0) / B1  (con el mismo B1 seguro del forward)
        b1_safe = pl.when((pl.col("B1") == 0) | pl.col("B1").is_null()).then(1.0).otherwise(pl.col("B1"))
        valor = pl.col(col_norm) * b1_safe + pl.col("B0")
    else:
        raise ValueError(f"metodo no soportado: {metodo}")
    return df.with_columns(valor.alias(alias))



### Agregar deltas (Saltos de k, es decir calcula las diferencias tn_i-tn_(i+k))

In [6]:
def agregar_deltas(df_norm, salto=2, prefix="", clase_col="clase_tn"):
    """
    Agrega columnas de delta (saltos de 'salto' periodos) sobre las columnas
    NORMALIZADAS. No elimina ninguna columna, solo agrega.

      {prefix}tn{k}_delta = {prefix}tn{k}_norm - {prefix}tn{k+salto}_norm  (k = 0 .. maxlag-salto)
      {clase_col}_delta   = {clase_col}_norm - {prefix}tn0_norm            (cambio a 2 periodos -> target)

    tn0_delta es el delta t0 - t2 (el cambio en el ti mas reciente).
    Para este problema salto=2 porque se predice a 2 periodos.

    Params
    ------
    prefix:
      prefijo de las columnas normalizadas a deltear (ej. "" para pc, "p_" para producto).
    clase_col:
      nombre base de la clase a deltear; None si no hay clase (ej. nivel producto).
    """
    p = len(prefix)
    norm_lags = sorted(
        int(c[p + 2:-5]) for c in df_norm.columns
        if c.startswith(prefix + "tn") and c.endswith("_norm") and c[p + 2:-5].isdigit()
    )
    max_k = max(norm_lags)
    exprs = [
        (pl.col(f"{prefix}tn{k}_norm") - pl.col(f"{prefix}tn{k + salto}_norm")).alias(f"{prefix}tn{k}_delta")
        for k in range(0, max_k - salto + 1)
    ]
    # Delta de la clase: valor de la clase normalizada menos t0 normalizado
    if clase_col is not None:
        exprs.append((pl.col(f"{clase_col}_norm") - pl.col(f"{prefix}tn0_norm")).alias(f"{clase_col}_delta"))
    return df_norm.with_columns(exprs)



### Agregar deltas (Calcula diferencias tn_0-tn_i)

In [108]:
def agregar_deltas2(df_norm, prefix=""):
    """
    Agrega columnas de delta calculadas como la diferencia entre tn0_norm 
    y cada tnk_norm (k >= 1) sobre las columnas NORMALIZADAS. 
    No elimina ninguna columna, solo agrega.

      {prefix}tn_delta{k} = {prefix}tn0_norm - {prefix}tn{k}_norm  (k = 1 .. maxlag)

    Params
    ------
    prefix:
      prefijo de las columnas normalizadas a deltear (ej. "" para pc, "p_" para producto).
    """
    p = len(prefix)
    norm_lags = sorted(
        int(c[p + 2:-5]) for c in df_norm.columns
        if c.startswith(prefix + "tn") and c.endswith("_norm") and c[p + 2:-5].isdigit()
    )
    max_k = max(norm_lags)
    exprs = [
        (pl.col(f"{prefix}tn0_norm") - pl.col(f"{prefix}tn{k}_norm")).alias(f"{prefix}tn_delta{k}")
        for k in range(1, max_k + 1)
    ]
    
    return df_norm.with_columns(exprs)

### Agregar rachas

In [7]:
def agregar_racha(df, prefix="", alias=None, suffix="_delta"):
    """
    Agrega una columna de RACHA: hace cuantos periodos seguidos viene subiendo (+)
    o bajando (-) la serie. Se lee sobre el signo de las columnas
    {prefix}tn*{suffix} que ya dejo agregar_deltas. No elimina nada, solo agrega.

      {prefix}tn_racha = direccion * largo

    Arranca en {prefix}tn0{suffix} (el cambio mas reciente) y avanza hacia los lags
    mas viejos mientras el signo no se de vuelta:
      - el signo del primer delta != 0 fija la direccion de la racha
      - un delta de signo contrario la corta
      - un delta = 0 (empate) NO la corta: la racha sigue viva y el empate suma
      - un delta nulo (el par todavia no existia) la corta

    Casos borde:
      - todos los deltas en 0 (serie plana, tipico de un producto sin ventas)
        -> racha = 0: no hay direccion.
      - {prefix}tn0{suffix} nulo (el par no llega a salto+1 periodos de historia)
        -> racha = null, para no confundir "sin historia" con "plano".
      - como el empate no corta, una serie que sube una vez y despues queda plana
        acumula racha alta (ej. deltas [1, 0, 0, 0] -> racha 4).

    Ojo: la unidad de la racha es el salto de agregar_deltas, no el mes. Con
    salto=2, racha=+3 son 3 deltas seguidos subiendo, o sea 6 meses de ventana.

    Params
    ------
    prefix : prefijo de las columnas a leer (ej. "" para pc, "p_" para producto,
             "c1_"/"c2_"/"c3_" para categoria).
    alias  : nombre de la columna de salida. Default: "{prefix}tn_racha".
    suffix : sufijo de las columnas a leer. Default "_delta".
    """
    p, s = len(prefix), len(suffix)
    ks = sorted(
        int(c[p + 2:-s]) for c in df.columns
        if c.startswith(f"{prefix}tn") and c.endswith(suffix) and c[p + 2:-s].isdigit()
    )
    if not ks:
        raise ValueError(f"no hay columnas {prefix}tn*{suffix} en el df")
    if alias is None:
        alias = f"{prefix}tn_racha"

    sgn = {k: pl.col(f"{prefix}tn{k}{suffix}").sign() for k in ks}

    # Direccion: signo del primer delta distinto de 0. Los nulls son de cola (los
    # meses previos al first_sell del par caen en los lags mas viejos), asi que el
    # primer delta != 0 siempre cae dentro del tramo con datos.
    # Sin direccion (todo empates, o todo nulo) -> 0, y la racha termina en 0.
    d = pl.coalesce([pl.when(sgn[k] != 0).then(sgn[k]) for k in ks]).fill_null(0).cast(pl.Int32)

    # Corta en el primer lag nulo o de signo contrario. sgn*d < 0 da falso para los
    # empates (producto 0) y para d = 0, que es justo lo que queremos.
    corta = {k: sgn[k].is_null() | ((sgn[k] * d) < 0) for k in ks}

    # Largo = indice del primer corte; si nunca corta, entra toda la ventana.
    largo = pl.min_horizontal([
        pl.when(corta[k]).then(pl.lit(k, dtype=pl.Int32)).otherwise(pl.lit(len(ks), dtype=pl.Int32))
        for k in ks
    ])

    return df.with_columns(
        pl.when(pl.col(f"{prefix}tn0{suffix}").is_null())
        .then(None)
        .otherwise(d * largo)
        .cast(pl.Int32)
        .alias(alias)
    )



## Armado de dataset

In [ ]:
### Todos los parámetros a configurar para el dataset final

### Estos 3 parámetros son los que definen el feature engineering histórico tanto al nivel de la granularidad como de agregaciones como producto, categorías padre o marca.

# Lags
MAX_LAGS = 24  # Cantidad máxima de lags a generar
# Normalización
METODO = "recta"  # Metodo de normalizacion: "recta"|"zscore"|"minmax"|"media"
# Deltas
SALTO = 2  # Cantidad de periodos de salto para calcular los deltas (1° versión)

In [ ]:
# Nos quedamos con los primeros 10 productos y los primeros 10 clientes para testear el pipeline
# df_final = df_final.filter(
#     (pl.col("product_id") <= 20010) & (pl.col("customer_id") <= 10010)
# )

In [111]:
# --- Uso ---
# Definir granularidad en funcion del nombre del dataset
granularidad = "pc" if "grpClienteProducto" in dataset else "p"

# Parametros del panel supervisado (compartidos con producto y categoria mas abajo):
#   PERIODO_FINAL : ultimo "periodo actual" (mes desde el que se predice).
#   N_PERIODOS    : cuantos periodos actuales generar hacia atras (rango).
#   MAX_LAGS      : lags tn0..tn{MAX_LAGS} por periodo.
#   HORIZONTE     : la clase es tn(periodo + HORIZONTE).
PERIODO_FINAL = 201912 # dejar en 201912, la division de train/val/test se hace en el modulo 3
N_PERIODOS    = 42
# MAX_LAGS    = 24  (Definido arriba)
HORIZONTE     = 2

# Una fila por (clave, periodo) para los ultimos N_PERIODOS meses hasta PERIODO_FINAL.
df_lags = lagear_dataset(
    df_final, granularidad,
    periodo_final=PERIODO_FINAL, n_periodos=N_PERIODOS,
    max_lags=MAX_LAGS, horizonte=HORIZONTE,
)
print(df_lags)

shape: (3_456, 35)
┌───────────────┬──────┬─────────────┬─────────┬───┬──────────┬──────────┬──────┬───────────┐
│ Agrupacion_ID ┆ cat1 ┆ cat2        ┆ cat3    ┆ … ┆ tn23     ┆ tn24     ┆ ppc0 ┆ clase_tn  │
│ ---           ┆ ---  ┆ ---         ┆ ---     ┆   ┆ ---      ┆ ---      ┆ ---  ┆ ---       │
│ i64           ┆ str  ┆ str         ┆ str     ┆   ┆ f64      ┆ f64      ┆ i64  ┆ f64       │
╞═══════════════╪══════╪═════════════╪═════════╪═══╪══════════╪══════════╪══════╪═══════════╡
│ 1000120001    ┆ HC   ┆ ROPA LAVADO ┆ Liquido ┆ … ┆ null     ┆ null     ┆ 0    ┆ 92.46537  │
│ 1000120001    ┆ HC   ┆ ROPA LAVADO ┆ Liquido ┆ … ┆ null     ┆ null     ┆ 0    ┆ 13.29728  │
│ 1000120001    ┆ HC   ┆ ROPA LAVADO ┆ Liquido ┆ … ┆ null     ┆ null     ┆ 0    ┆ 101.00563 │
│ 1000120001    ┆ HC   ┆ ROPA LAVADO ┆ Liquido ┆ … ┆ null     ┆ null     ┆ 0    ┆ 128.04792 │
│ 1000120001    ┆ HC   ┆ ROPA LAVADO ┆ Liquido ┆ … ┆ null     ┆ null     ┆ 0    ┆ 101.20711 │
│ …             ┆ …    ┆ …           ┆ … 

In [112]:
df_lags.columns

['Agrupacion_ID',
 'cat1',
 'cat2',
 'cat3',
 'brand',
 'product_id',
 'customer_id',
 'periodo',
 'tn0',
 'tn1',
 'tn2',
 'tn3',
 'tn4',
 'tn5',
 'tn6',
 'tn7',
 'tn8',
 'tn9',
 'tn10',
 'tn11',
 'tn12',
 'tn13',
 'tn14',
 'tn15',
 'tn16',
 'tn17',
 'tn18',
 'tn19',
 'tn20',
 'tn21',
 'tn22',
 'tn23',
 'tn24',
 'ppc0',
 'clase_tn']

In [113]:
# --- Uso ---
# metodo: "recta" | "zscore" | "minmax" | "media"
# norm_lags: ventana tn0..tn{norm_lags} para calcular B0/B1 (None = todos los lags)
df_norm = normalizar_achatado(df_lags, metodo=METODO, norm_lags=None)
print(df_norm)

shape: (3_456, 63)
┌───────────────┬──────┬────────┬─────────┬───┬───────────┬────────────┬───────────┬───────────────┐
│ Agrupacion_ID ┆ cat1 ┆ cat2   ┆ cat3    ┆ … ┆ tn22_norm ┆ tn23_norm  ┆ tn24_norm ┆ clase_tn_norm │
│ ---           ┆ ---  ┆ ---    ┆ ---     ┆   ┆ ---       ┆ ---        ┆ ---       ┆ ---           │
│ i64           ┆ str  ┆ str    ┆ str     ┆   ┆ f64       ┆ f64        ┆ f64       ┆ f64           │
╞═══════════════╪══════╪════════╪═════════╪═══╪═══════════╪════════════╪═══════════╪═══════════════╡
│ 1000120001    ┆ HC   ┆ ROPA   ┆ Liquido ┆ … ┆ null      ┆ null       ┆ null      ┆ -6.97324      │
│               ┆      ┆ LAVADO ┆         ┆   ┆           ┆            ┆           ┆               │
│ 1000120001    ┆ HC   ┆ ROPA   ┆ Liquido ┆ … ┆ null      ┆ null       ┆ null      ┆ -384.35645    │
│               ┆      ┆ LAVADO ┆         ┆   ┆           ┆            ┆           ┆               │
│ 1000120001    ┆ HC   ┆ ROPA   ┆ Liquido ┆ … ┆ null      ┆ null       ┆

In [114]:

# --- Uso ---
# Chequeo round-trip: desnormalizar clase_tn_norm (lag=-2) tiene que devolver clase_tn.
df_check = desnormalizar(df_norm, "clase_tn_norm", METODO, lag=-2, alias="clase_tn_rec")
error_max = df_check.select((pl.col("clase_tn_rec") - pl.col("clase_tn")).abs().max()).item()
print("Error maximo de round-trip:", error_max)

# Cuando tengas la prediccion del modelo en escala normalizada (ej. columna 'clase_tn_norm_pred'),
# la pasas a toneladas asi:
# df_pred = desnormalizar(df_pred, "clase_tn_norm_pred", METODO, lag=-2, alias="tn_pred")


Error maximo de round-trip: 2.842170943040401e-14


In [115]:

# --- Uso ---
df_norm = agregar_deltas(df_norm, salto=SALTO)
print(df_norm)


shape: (3_456, 87)
┌──────────────┬──────┬────────┬─────────┬───┬────────────┬────────────┬────────────┬──────────────┐
│ Agrupacion_I ┆ cat1 ┆ cat2   ┆ cat3    ┆ … ┆ tn20_delta ┆ tn21_delta ┆ tn22_delta ┆ clase_tn_del │
│ D            ┆ ---  ┆ ---    ┆ ---     ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ta           │
│ ---          ┆ str  ┆ str    ┆ str     ┆   ┆ f64        ┆ f64        ┆ f64        ┆ ---          │
│ i64          ┆      ┆        ┆         ┆   ┆            ┆            ┆            ┆ f64          │
╞══════════════╪══════╪════════╪═════════╪═══╪════════════╪════════════╪════════════╪══════════════╡
│ 1000120001   ┆ HC   ┆ ROPA   ┆ Liquido ┆ … ┆ null       ┆ null       ┆ null       ┆ -6.97324     │
│              ┆      ┆ LAVADO ┆         ┆   ┆            ┆            ┆            ┆              │
│ 1000120001   ┆ HC   ┆ ROPA   ┆ Liquido ┆ … ┆ null       ┆ null       ┆ null       ┆ -384.35645   │
│              ┆      ┆ LAVADO ┆         ┆   ┆            ┆            ┆

In [116]:
df_norm.columns

['Agrupacion_ID',
 'cat1',
 'cat2',
 'cat3',
 'brand',
 'product_id',
 'customer_id',
 'periodo',
 'tn0',
 'tn1',
 'tn2',
 'tn3',
 'tn4',
 'tn5',
 'tn6',
 'tn7',
 'tn8',
 'tn9',
 'tn10',
 'tn11',
 'tn12',
 'tn13',
 'tn14',
 'tn15',
 'tn16',
 'tn17',
 'tn18',
 'tn19',
 'tn20',
 'tn21',
 'tn22',
 'tn23',
 'tn24',
 'ppc0',
 'clase_tn',
 'B0',
 'B1',
 'tn0_norm',
 'tn1_norm',
 'tn2_norm',
 'tn3_norm',
 'tn4_norm',
 'tn5_norm',
 'tn6_norm',
 'tn7_norm',
 'tn8_norm',
 'tn9_norm',
 'tn10_norm',
 'tn11_norm',
 'tn12_norm',
 'tn13_norm',
 'tn14_norm',
 'tn15_norm',
 'tn16_norm',
 'tn17_norm',
 'tn18_norm',
 'tn19_norm',
 'tn20_norm',
 'tn21_norm',
 'tn22_norm',
 'tn23_norm',
 'tn24_norm',
 'clase_tn_norm',
 'tn0_delta',
 'tn1_delta',
 'tn2_delta',
 'tn3_delta',
 'tn4_delta',
 'tn5_delta',
 'tn6_delta',
 'tn7_delta',
 'tn8_delta',
 'tn9_delta',
 'tn10_delta',
 'tn11_delta',
 'tn12_delta',
 'tn13_delta',
 'tn14_delta',
 'tn15_delta',
 'tn16_delta',
 'tn17_delta',
 'tn18_delta',
 'tn19_delta',
 't

In [117]:
#checkeo
_cols = [c for c in ["product_id", "customer_id", "periodo", "tn0_norm", "tn1_norm",
                     "tn2_norm", "tn3_norm", "tn0_delta", "tn1_delta",
                     "clase_tn_norm", "clase_tn_delta"] if c in df_norm.columns]
df_norm.select(_cols)

product_id,customer_id,periodo,tn0_norm,tn1_norm,tn2_norm,tn3_norm,tn0_delta,tn1_delta,clase_tn_norm,clase_tn_delta
i64,i64,i32,f64,f64,f64,f64,f64,f64,f64,f64
20001,10001,201701,0.0,null,null,null,null,null,-6.97324,-6.97324
20001,10001,201702,-2.8422e-14,-1.4211e-14,null,null,null,null,-384.35645,-384.35645
20001,10001,201703,-34.29722,68.59444,-34.29722,null,-4.2633e-14,null,-18.78372,15.5135
20001,10001,201704,-32.993607,9.694256,79.592309,-56.292958,-112.585916,65.987214,154.717487,187.711094
20001,10001,201705,36.477988,-69.471595,-8.544738,79.592309,45.022726,-149.063904,73.161934,36.683946
…,…,…,…,…,…,…,…,…,…,…
20010,10010,201908,12.364074,12.786564,9.015765,-5.657555,3.348309,18.444119,24.098943,11.734869
20010,10010,201909,10.585883,8.095456,8.959658,5.630571,1.626224,2.464884,8.857487,-1.728396
20010,10010,201910,16.737505,8.470481,6.074046,7.032241,10.663459,1.438239,-11.177216,-27.914721


In [118]:
# --- Uso ---
df_norm = agregar_deltas2(df_norm)
print(df_norm)

shape: (3_456, 111)
┌────────────────┬──────┬────────┬─────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ Agrupacion_ID  ┆ cat1 ┆ cat2   ┆ cat3    ┆ … ┆ tn_delta21 ┆ tn_delta22 ┆ tn_delta23 ┆ tn_delta24 │
│ ---            ┆ ---  ┆ ---    ┆ ---     ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│ i64            ┆ str  ┆ str    ┆ str     ┆   ┆ f64        ┆ f64        ┆ f64        ┆ f64        │
╞════════════════╪══════╪════════╪═════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 1000120001     ┆ HC   ┆ ROPA   ┆ Liquido ┆ … ┆ null       ┆ null       ┆ null       ┆ null       │
│                ┆      ┆ LAVADO ┆         ┆   ┆            ┆            ┆            ┆            │
│ 1000120001     ┆ HC   ┆ ROPA   ┆ Liquido ┆ … ┆ null       ┆ null       ┆ null       ┆ null       │
│                ┆      ┆ LAVADO ┆         ┆   ┆            ┆            ┆            ┆            │
│ 1000120001     ┆ HC   ┆ ROPA   ┆ Liquido ┆ … ┆ null       ┆ null     

Agrego a la granularidad pc, la de p para complementar el dataset

In [ ]:
df_aux = pl.read_parquet(Ruta+dataset)

In [120]:
df_aux = df_aux.with_columns(
    (pl.col("periodo").dt.year() * 100 + pl.col("periodo").dt.month()).alias("periodo")
)

df_aux = df_aux.with_columns(
        pl.col("periodo").min().over("product_id").alias("first_sell_in_period")
    )

print(df_aux)

shape: (10_792_694, 15)
┌─────────────┬────────────┬────────────┬─────────┬───┬───────┬──────────┬────────────┬────────────┐
│ Agrupacion_ ┆ customer_i ┆ product_id ┆ periodo ┆ … ┆ brand ┆ sku_size ┆ descripcio ┆ first_sell │
│ ID          ┆ d          ┆ ---        ┆ ---     ┆   ┆ ---   ┆ ---      ┆ n          ┆ _in_period │
│ ---         ┆ ---        ┆ i64        ┆ i32     ┆   ┆ str   ┆ i64      ┆ ---        ┆ ---        │
│ i64         ┆ i64        ┆            ┆         ┆   ┆       ┆          ┆ str        ┆ i32        │
╞═════════════╪════════════╪════════════╪═════════╪═══╪═══════╪══════════╪════════════╪════════════╡
│ 1004620876  ┆ 10046      ┆ 20876      ┆ 201701  ┆ … ┆ NIVEA ┆ 200      ┆ Cabello    ┆ 201701     │
│             ┆            ┆            ┆         ┆   ┆       ┆          ┆ anti caída ┆            │
│ 1004620876  ┆ 10046      ┆ 20876      ┆ 201702  ┆ … ┆ NIVEA ┆ 200      ┆ Cabello    ┆ 201701     │
│             ┆            ┆            ┆         ┆   ┆       ┆    

In [121]:
# Colapsamos la dimension cliente: agregamos por periodo y producto.
#   - cust_request_qty, cust_request_tn, tn -> suma (acumulado sobre los clientes)
#   - first_sell_in_period                 -> minimo del grupo
# plan_precios_cuidados NO se agrega a nivel producto: nos quedamos solo con la
# columna original (nivel producto-cliente), que es la mas realista.
df_aux = (
    df_aux
    .group_by(["periodo", "product_id"])
    .agg(
        pl.col("cust_request_qty").sum().alias("cust_request_qty"),
        pl.col("cust_request_tn").sum().alias("cust_request_tn"),
        pl.col("tn").sum().alias("tn"),
        pl.col("first_sell_in_period").min().alias("first_sell_in_period"),
        pl.col("cat1").first().alias("cat1"),
        pl.col("cat2").first().alias("cat2"),
        pl.col("cat3").first().alias("cat3"),
        pl.col("brand").first().alias("brand"),
    )
    .sort(["product_id", "periodo"])
)
print(df_aux)

shape: (31_522, 10)
┌─────────┬────────────┬────────────────┬───────────────┬───┬──────┬─────────────┬─────────┬───────┐
│ periodo ┆ product_id ┆ cust_request_q ┆ cust_request_ ┆ … ┆ cat1 ┆ cat2        ┆ cat3    ┆ brand │
│ ---     ┆ ---        ┆ ty             ┆ tn            ┆   ┆ ---  ┆ ---         ┆ ---     ┆ ---   │
│ i32     ┆ i64        ┆ ---            ┆ ---           ┆   ┆ str  ┆ str         ┆ str     ┆ str   │
│         ┆            ┆ i64            ┆ f64           ┆   ┆      ┆             ┆         ┆       │
╞═════════╪════════════╪════════════════╪═══════════════╪═══╪══════╪═════════════╪═════════╪═══════╡
│ 201701  ┆ 20001      ┆ 479            ┆ 937.72717     ┆ … ┆ HC   ┆ ROPA LAVADO ┆ Liquido ┆ ARIEL │
│ 201702  ┆ 20001      ┆ 432            ┆ 833.72187     ┆ … ┆ HC   ┆ ROPA LAVADO ┆ Liquido ┆ ARIEL │
│ 201703  ┆ 20001      ┆ 509            ┆ 1330.74697    ┆ … ┆ HC   ┆ ROPA LAVADO ┆ Liquido ┆ ARIEL │
│ 201704  ┆ 20001      ┆ 279            ┆ 1132.9443     ┆ … ┆ HC   ┆ RO

In [122]:
# --- Features a nivel PRODUCTO (complemento "p" de la granularidad pc) ---
# Solo tiene sentido con granularidad "pc": ahi cada fila es un par producto-cliente
# y el total del producto (tn sumado sobre TODOS sus clientes) es una senial NUEVA
# que complementa la serie del par. Con granularidad "p" el dataset YA viene agregado
# por producto -> tn* == p_tn*, asi que este complemento seria redundante y lo salteamos.
if granularidad == "pc":
    # Mismo panel (mismos PERIODO_FINAL / N_PERIODOS) pero a nivel producto, sumando tn
    # sobre TODOS los clientes. Sale de df_aux (mercado completo), NO de df_final
    # (filtrado) -> ahi el total por producto seria parcial. Se pega por
    # product_id + periodo (cada periodo actual del panel).
    prod_feats = lagear_dataset(
        df_aux, granularidad="p",
        periodo_final=PERIODO_FINAL, n_periodos=N_PERIODOS,
        max_lags=MAX_LAGS, horizonte=HORIZONTE,
    )

    lag_cols_p = [c for c in prod_feats.columns if c.startswith("tn") and c[2:].isdigit()]
    prod_feats = (
        prod_feats
        .drop(["clase_tn", "cat1", "cat2", "cat3"])
        # tn{k} -> p_tn{k} para no pisar los tn{k} de nivel producto-cliente
        .rename({c: f"p_{c}" for c in lag_cols_p})
        # mes sin ventas del producto -> 0 tn (a nivel producto un mes ausente = 0)
        .with_columns([pl.col(f"p_{c}").fill_null(0.0) for c in lag_cols_p])
    )

    # Pegamos las features de producto a cada fila (producto-cliente) de df_norm
    df_norm = df_norm.join(prod_feats, on=["product_id", "periodo"], how="left")

    print(df_norm.select(
        ["product_id", "customer_id", "periodo"] + [f"p_tn{k}" for k in range(MAX_LAGS + 1)]
    ))
else:
    print(f"granularidad={granularidad!r}: sin complemento a nivel producto (tn* ya es producto).")

shape: (3_456, 28)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ product_id ┆ customer_ ┆ periodo ┆ p_tn0     ┆ … ┆ p_tn21    ┆ p_tn22    ┆ p_tn23    ┆ p_tn24    │
│ ---        ┆ id        ┆ ---     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ i64        ┆ ---       ┆ i32     ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
│            ┆ i64       ┆         ┆           ┆   ┆           ┆           ┆           ┆           │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20001      ┆ 10001     ┆ 201701  ┆ 934.77222 ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ 20001      ┆ 10001     ┆ 201702  ┆ 798.0162  ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ 20001      ┆ 10001     ┆ 201703  ┆ 1303.3577 ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│            ┆           ┆         ┆ 1         ┆   ┆           ┆        

In [123]:
df_norm.columns

['Agrupacion_ID',
 'cat1',
 'cat2',
 'cat3',
 'brand',
 'product_id',
 'customer_id',
 'periodo',
 'tn0',
 'tn1',
 'tn2',
 'tn3',
 'tn4',
 'tn5',
 'tn6',
 'tn7',
 'tn8',
 'tn9',
 'tn10',
 'tn11',
 'tn12',
 'tn13',
 'tn14',
 'tn15',
 'tn16',
 'tn17',
 'tn18',
 'tn19',
 'tn20',
 'tn21',
 'tn22',
 'tn23',
 'tn24',
 'ppc0',
 'clase_tn',
 'B0',
 'B1',
 'tn0_norm',
 'tn1_norm',
 'tn2_norm',
 'tn3_norm',
 'tn4_norm',
 'tn5_norm',
 'tn6_norm',
 'tn7_norm',
 'tn8_norm',
 'tn9_norm',
 'tn10_norm',
 'tn11_norm',
 'tn12_norm',
 'tn13_norm',
 'tn14_norm',
 'tn15_norm',
 'tn16_norm',
 'tn17_norm',
 'tn18_norm',
 'tn19_norm',
 'tn20_norm',
 'tn21_norm',
 'tn22_norm',
 'tn23_norm',
 'tn24_norm',
 'clase_tn_norm',
 'tn0_delta',
 'tn1_delta',
 'tn2_delta',
 'tn3_delta',
 'tn4_delta',
 'tn5_delta',
 'tn6_delta',
 'tn7_delta',
 'tn8_delta',
 'tn9_delta',
 'tn10_delta',
 'tn11_delta',
 'tn12_delta',
 'tn13_delta',
 'tn14_delta',
 'tn15_delta',
 'tn16_delta',
 'tn17_delta',
 'tn18_delta',
 'tn19_delta',
 't

In [ ]:
# Normalizamos los lags de PRODUCTO con la misma funcion/metodo que los pc,
# pero referenciados al producto: parametros propios p_B0 / p_B1 y columnas
# p_tn*_norm. Sin clase (clase_col=None): a nivel producto no normalizamos ninguna.
# metodo debe ser el mismo que en la normalizacion pc de arriba ("media").
# Solo con granularidad "pc": las columnas p_tn* solo existen en ese caso.
if granularidad == "pc":
    df_norm = normalizar_achatado(
        df_norm, metodo=METODO, norm_lags=None,
        prefix="p_", clase_col=None, b0_name="p_B0", b1_name="p_B1",
    )

    print(df_norm.select(
        ["product_id", "customer_id", "periodo", "p_B0", "p_B1"]
        + [f"p_tn{k}_norm" for k in range(MAX_LAGS + 1)]
    ))

shape: (3_456, 30)
┌────────────┬────────────┬─────────┬──────┬───┬────────────┬────────────┬────────────┬────────────┐
│ product_id ┆ customer_i ┆ periodo ┆ p_B0 ┆ … ┆ p_tn21_nor ┆ p_tn22_nor ┆ p_tn23_nor ┆ p_tn24_nor │
│ ---        ┆ d          ┆ ---     ┆ ---  ┆   ┆ m          ┆ m          ┆ m          ┆ m          │
│ i64        ┆ ---        ┆ i32     ┆ f64  ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│            ┆ i64        ┆         ┆      ┆   ┆ f64        ┆ f64        ┆ f64        ┆ f64        │
╞════════════╪════════════╪═════════╪══════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 20001      ┆ 10001      ┆ 201701  ┆ 0.0  ┆ … ┆ 0.0        ┆ 0.0        ┆ 0.0        ┆ 0.0        │
│ 20001      ┆ 10001      ┆ 201702  ┆ 0.0  ┆ … ┆ 0.0        ┆ 0.0        ┆ 0.0        ┆ 0.0        │
│ 20001      ┆ 10001      ┆ 201703  ┆ 0.0  ┆ … ┆ 0.0        ┆ 0.0        ┆ 0.0        ┆ 0.0        │
│ 20001      ┆ 10001      ┆ 201704  ┆ 0.0  ┆ … ┆ 0.0        ┆ 0.0       

In [ ]:
# Deltas de los lags de PRODUCTO: misma idea que los pc pero sobre p_tn*_norm.
#   p_tn{k}_delta = p_tn{k}_norm - p_tn{k+2}_norm
# Sin clase (clase_col=None): la clase vive solo a nivel producto-cliente.
# Solo con granularidad "pc": las columnas p_tn*_norm solo existen en ese caso.
if granularidad == "pc":
    df_norm = agregar_deltas(df_norm, salto=SALTO, prefix="p_", clase_col=None)

    print(df_norm.select(
        ["product_id", "customer_id", "periodo"]
        + [f"p_tn{k}_norm" for k in range(MAX_LAGS + 1)]
        + [f"p_tn{k}_delta" for k in range(MAX_LAGS - 1)]
    ))

shape: (3_456, 51)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ product_id ┆ customer_ ┆ periodo ┆ p_tn0_nor ┆ … ┆ p_tn19_de ┆ p_tn20_de ┆ p_tn21_de ┆ p_tn22_de │
│ ---        ┆ id        ┆ ---     ┆ m         ┆   ┆ lta       ┆ lta       ┆ lta       ┆ lta       │
│ i64        ┆ ---       ┆ i32     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ i64       ┆         ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20001      ┆ 10001     ┆ 201701  ┆ 25.0      ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ 20001      ┆ 10001     ┆ 201702  ┆ 11.513469 ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ 20001      ┆ 10001     ┆ 201703  ┆ 10.732007 ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ 20001      ┆ 10001     ┆ 201704  ┆ 6.51445   ┆ … ┆ 0.0       ┆ 0.0    

In [136]:
# Deltas de los lags de PRODUCTO respecto al momento 0
if granularidad == "pc":
    df_norm = agregar_deltas2(df_norm, prefix="p_")

    print(df_norm.select(
        ["product_id", "customer_id", "periodo"]
        + [f"p_tn{k}_norm" for k in range(MAX_LAGS + 1)]
        + [f"p_tn_delta{k}" for k in range(1,MAX_LAGS - 1)]
    ))

shape: (3_456, 50)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ product_id ┆ customer_ ┆ periodo ┆ p_tn0_nor ┆ … ┆ p_tn_delt ┆ p_tn_delt ┆ p_tn_delt ┆ p_tn_delt │
│ ---        ┆ id        ┆ ---     ┆ m         ┆   ┆ a19       ┆ a20       ┆ a21       ┆ a22       │
│ i64        ┆ ---       ┆ i32     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ i64       ┆         ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20001      ┆ 10001     ┆ 201701  ┆ 25.0      ┆ … ┆ 25.0      ┆ 25.0      ┆ 25.0      ┆ 25.0      │
│ 20001      ┆ 10001     ┆ 201702  ┆ 11.513469 ┆ … ┆ 11.513469 ┆ 11.513469 ┆ 11.513469 ┆ 11.513469 │
│ 20001      ┆ 10001     ┆ 201703  ┆ 10.732007 ┆ … ┆ 10.732007 ┆ 10.732007 ┆ 10.732007 ┆ 10.732007 │
│ 20001      ┆ 10001     ┆ 201704  ┆ 6.51445   ┆ … ┆ 6.51445   ┆ 6.51445

In [137]:
len(df_norm.columns)

211

In [138]:
df_norm.columns

['Agrupacion_ID',
 'cat1',
 'cat2',
 'cat3',
 'brand',
 'product_id',
 'customer_id',
 'periodo',
 'tn0',
 'tn1',
 'tn2',
 'tn3',
 'tn4',
 'tn5',
 'tn6',
 'tn7',
 'tn8',
 'tn9',
 'tn10',
 'tn11',
 'tn12',
 'tn13',
 'tn14',
 'tn15',
 'tn16',
 'tn17',
 'tn18',
 'tn19',
 'tn20',
 'tn21',
 'tn22',
 'tn23',
 'tn24',
 'ppc0',
 'clase_tn',
 'B0',
 'B1',
 'tn0_norm',
 'tn1_norm',
 'tn2_norm',
 'tn3_norm',
 'tn4_norm',
 'tn5_norm',
 'tn6_norm',
 'tn7_norm',
 'tn8_norm',
 'tn9_norm',
 'tn10_norm',
 'tn11_norm',
 'tn12_norm',
 'tn13_norm',
 'tn14_norm',
 'tn15_norm',
 'tn16_norm',
 'tn17_norm',
 'tn18_norm',
 'tn19_norm',
 'tn20_norm',
 'tn21_norm',
 'tn22_norm',
 'tn23_norm',
 'tn24_norm',
 'clase_tn_norm',
 'tn0_delta',
 'tn1_delta',
 'tn2_delta',
 'tn3_delta',
 'tn4_delta',
 'tn5_delta',
 'tn6_delta',
 'tn7_delta',
 'tn8_delta',
 'tn9_delta',
 'tn10_delta',
 'tn11_delta',
 'tn12_delta',
 'tn13_delta',
 'tn14_delta',
 'tn15_delta',
 'tn16_delta',
 'tn17_delta',
 'tn18_delta',
 'tn19_delta',
 't

In [139]:
# --- Toneladas de la jerarquia (cat1, cat2, cat3) y por brand por periodo ---
# Tamanio de mercado CONTEMPORANEO: para cada periodo actual t, el tn total de la
# categoria en t (incluye al propio producto). Una feature por nivel.
#
# Se calcula sobre df_aux (mercado completo, tn ya sumado sobre los clientes), NO
# sobre df_final (filtrado) -> ahi el total de la categoria seria parcial.

CATS = ["cat1", "cat2", "cat3", "brand"]

# cat1/cat2/cat3 ya vienen en df_norm desde el panel base (lagear_dataset las arrastra
# como metadata 1:1 del producto), asi que no hace falta re-pegarlas aca.

# tn total de cada categoria en CADA periodo; lo pegamos por (categoria, periodo).
for c in CATS:
    tn_cat = df_aux.group_by([c, "periodo"]).agg(pl.col("tn").sum().alias(f"tn_total_{c}"))
    df_norm = df_norm.join(tn_cat, on=[c, "periodo"], how="left")

# Una categoria sin ventas en ese periodo vendio 0 tn; no es un dato faltante.
df_norm = df_norm.with_columns([pl.col(f"tn_total_{c}").fill_null(0.0) for c in CATS])

# customer_id solo existe con granularidad "pc" (en "p" el panel es por producto).
_id_cols = [c for c in ["product_id", "customer_id", "periodo"] if c in df_norm.columns]
print(df_norm.select(
    _id_cols + CATS + [f"tn_total_{c}" for c in CATS]
))

shape: (3_456, 11)
┌────────────┬────────────┬─────────┬──────┬───┬────────────┬────────────┬────────────┬────────────┐
│ product_id ┆ customer_i ┆ periodo ┆ cat1 ┆ … ┆ tn_total_c ┆ tn_total_c ┆ tn_total_c ┆ tn_total_b │
│ ---        ┆ d          ┆ ---     ┆ ---  ┆   ┆ at1        ┆ at2        ┆ at3        ┆ rand       │
│ i64        ┆ ---        ┆ i32     ┆ str  ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│            ┆ i64        ┆         ┆      ┆   ┆ f64        ┆ f64        ┆ f64        ┆ f64        │
╞════════════╪════════════╪═════════╪══════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 20001      ┆ 10001      ┆ 201701  ┆ HC   ┆ … ┆ 20304.2869 ┆ 11153.2960 ┆ 3871.38669 ┆ 2655.78679 │
│            ┆            ┆         ┆      ┆   ┆ 6          ┆ 7          ┆            ┆            │
│ 20001      ┆ 10001      ┆ 201702  ┆ HC   ┆ … ┆ 20183.3015 ┆ 10799.0724 ┆ 3908.83266 ┆ 2213.71985 │
│            ┆            ┆         ┆      ┆   ┆ 2          ┆           

In [ ]:
# --- Panel por CATEGORIA (cat1, cat2, cat3) ---
# Mismo pipeline que el nivel producto pero con keys=[cat]: reusamos las tres
# funciones tal cual (lagear -> normalizar -> deltas), solo cambia el prefijo.
#
# Los tres niveles aportan: cat3 agrupa 5 productos en mediana (max 134), asi que su
# serie no es la del producto suelto. Solo 12 de los 1233 productos (1%) son unicos en
# su cat3; para esos c3_tn* si termina siendo igual a p_tn*.

NIVELES = [("cat1", "c1_"), ("cat2", "c2_"), ("cat3", "c3_"), ("brand", "b_")]

for cat, pref in NIVELES:
    # Serie mensual de la categoria: tn de todo el mercado sumado sobre sus productos.
    # Sale de df_aux (mercado completo), no de df_final (filtrado).
    df_cat = (
        df_aux
        .group_by([cat, "periodo"])
        .agg(pl.col("tn").sum().alias("tn"))
    )

    # Lags de la categoria por periodo. Sin clase: la clase vive a nivel producto-cliente.
    cat_feats = lagear_dataset(
        df_cat, keys=[cat],
        periodo_final=PERIODO_FINAL, n_periodos=N_PERIODOS,
        max_lags=MAX_LAGS, horizonte=HORIZONTE,
    ).drop("clase_tn")
    lag_cols_c = [c for c in cat_feats.columns if c.startswith("tn") and c[2:].isdigit()]
    # tn{k} -> {pref}tn{k} para no pisar los tn{k} de nivel producto-cliente
    cat_feats = cat_feats.rename({c: f"{pref}{c}" for c in lag_cols_c})

    df_norm = df_norm.join(cat_feats, on=[cat, "periodo"], how="left")

    # Dos fuentes de null, y las dos significan 0 tn (mismo criterio que p_tn*):
    #   - mes sin ventas de la categoria -> el lag queda null
    #   - categoria sin fila en ese periodo del panel -> el left join no la encuentra
    # Rellenamos despues del join y antes de normalizar para que B0/B1 no salgan nulos.
    df_norm = df_norm.with_columns(
        [pl.col(f"{pref}tn{k}").fill_null(0.0) for k in range(MAX_LAGS + 1)]
    )

    df_norm = normalizar_achatado(
        df_norm, metodo=METODO, norm_lags=None,
        prefix=pref, clase_col=None, b0_name=f"{pref}B0", b1_name=f"{pref}B1",
    )
    df_norm = agregar_deltas(df_norm, salto=SALTO, prefix=pref, clase_col=None)

    df_norm = agregar_deltas2(df_norm, prefix=pref)

# customer_id solo existe con granularidad "pc" (en "p" el panel es por producto).
_id_cols = [c for c in ["product_id", "customer_id", "periodo", "cat1"] if c in df_norm.columns]
print(df_norm.select(
    _id_cols
    + [f"c1_tn{k}_norm" for k in range(3)]
    + [f"c1_tn{k}_delta" for k in range(2)]
    + [f"c1_tn_delta{k}" for k in range(1, 3)]
    + [f"c2_tn{k}_norm" for k in range(3)]
))

shape: (3_456, 14)
┌────────────┬────────────┬─────────┬──────┬───┬────────────┬────────────┬────────────┬────────────┐
│ product_id ┆ customer_i ┆ periodo ┆ cat1 ┆ … ┆ c1_tn_delt ┆ c2_tn0_nor ┆ c2_tn1_nor ┆ c2_tn2_nor │
│ ---        ┆ d          ┆ ---     ┆ ---  ┆   ┆ a2         ┆ m          ┆ m          ┆ m          │
│ i64        ┆ ---        ┆ i32     ┆ str  ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│            ┆ i64        ┆         ┆      ┆   ┆ f64        ┆ f64        ┆ f64        ┆ f64        │
╞════════════╪════════════╪═════════╪══════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 20001      ┆ 10001      ┆ 201701  ┆ HC   ┆ … ┆ 25.0       ┆ 25.0       ┆ 0.0        ┆ 0.0        │
│ 20001      ┆ 10001      ┆ 201702  ┆ HC   ┆ … ┆ 12.462647  ┆ 12.2983    ┆ 12.7017    ┆ 0.0        │
│ 20001      ┆ 10001      ┆ 201703  ┆ HC   ┆ … ┆ 3.091706   ┆ 10.372036  ┆ 7.195963   ┆ 7.432      │
│ 20001      ┆ 10001      ┆ 201704  ┆ HC   ┆ … ┆ 1.137757   ┆ 6.236827  

In [141]:
df_norm.columns

['Agrupacion_ID',
 'cat1',
 'cat2',
 'cat3',
 'brand',
 'product_id',
 'customer_id',
 'periodo',
 'tn0',
 'tn1',
 'tn2',
 'tn3',
 'tn4',
 'tn5',
 'tn6',
 'tn7',
 'tn8',
 'tn9',
 'tn10',
 'tn11',
 'tn12',
 'tn13',
 'tn14',
 'tn15',
 'tn16',
 'tn17',
 'tn18',
 'tn19',
 'tn20',
 'tn21',
 'tn22',
 'tn23',
 'tn24',
 'ppc0',
 'clase_tn',
 'B0',
 'B1',
 'tn0_norm',
 'tn1_norm',
 'tn2_norm',
 'tn3_norm',
 'tn4_norm',
 'tn5_norm',
 'tn6_norm',
 'tn7_norm',
 'tn8_norm',
 'tn9_norm',
 'tn10_norm',
 'tn11_norm',
 'tn12_norm',
 'tn13_norm',
 'tn14_norm',
 'tn15_norm',
 'tn16_norm',
 'tn17_norm',
 'tn18_norm',
 'tn19_norm',
 'tn20_norm',
 'tn21_norm',
 'tn22_norm',
 'tn23_norm',
 'tn24_norm',
 'clase_tn_norm',
 'tn0_delta',
 'tn1_delta',
 'tn2_delta',
 'tn3_delta',
 'tn4_delta',
 'tn5_delta',
 'tn6_delta',
 'tn7_delta',
 'tn8_delta',
 'tn9_delta',
 'tn10_delta',
 'tn11_delta',
 'tn12_delta',
 'tn13_delta',
 'tn14_delta',
 'tn15_delta',
 'tn16_delta',
 'tn17_delta',
 'tn18_delta',
 'tn19_delta',
 't

In [142]:

# --- Uso: una racha por nivel (producto-cliente, producto y las 3 categorias) ---
# El nivel "p_" (producto) solo existe con granularidad "pc"; ver el complemento a
# nivel producto de mas arriba. Con granularidad "p" no se genera.
RACHA_PREFIXES = ["", "c1_", "c2_", "c3_", "b_"]
if granularidad == "pc":
    RACHA_PREFIXES.insert(1, "p_")

for pref in RACHA_PREFIXES:
    df_norm = agregar_racha(df_norm, prefix=pref)

# customer_id solo existe con granularidad "pc" (en "p" el achatado es por producto).
_id_cols = [c for c in ["product_id", "customer_id", "periodo"] if c in df_norm.columns]
print(df_norm.select(
    _id_cols
    + [f"tn{k}_delta" for k in range(3)]
    + [f"{pref}tn_racha" for pref in RACHA_PREFIXES]
))

shape: (3_456, 12)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ product_id ┆ customer_ ┆ periodo ┆ tn0_delta ┆ … ┆ c1_tn_rac ┆ c2_tn_rac ┆ c3_tn_rac ┆ b_tn_rach │
│ ---        ┆ id        ┆ ---     ┆ ---       ┆   ┆ ha        ┆ ha        ┆ ha        ┆ a         │
│ i64        ┆ ---       ┆ i32     ┆ f64       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ i64       ┆         ┆           ┆   ┆ i32       ┆ i32       ┆ i32       ┆ i32       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20001      ┆ 10001     ┆ 201701  ┆ null      ┆ … ┆ 23        ┆ 23        ┆ 23        ┆ 23        │
│ 20001      ┆ 10001     ┆ 201702  ┆ null      ┆ … ┆ 23        ┆ 23        ┆ 23        ┆ 23        │
│ 20001      ┆ 10001     ┆ 201703  ┆ -4.2633e- ┆ … ┆ 23        ┆ 23        ┆ 23        ┆ 23        │
│            ┆           ┆         ┆ 14        ┆   ┆           ┆        

In [91]:
len(df_norm.columns)

501

In [92]:
df_norm.shape

(3456, 501)

In [ ]:
# Exportamos el dataset
df_norm.write_parquet(f"{Ruta}{dataset[:-8]}_{MAX_LAGS}lags_{METODO}_{SALTO}deltas.parquet")

FileNotFoundError: El sistema no puede encontrar la ruta especificada. (os error 3): /content/buckets/b1/datasets/df_procesado.parquet